# Colab：Drive 挂载 + `dataset` 目录结构粗检

1. 挂载 Google Drive  
2. 检查 `MyDrive/dataset/` 下各数据集文件夹（子文件夹仅看前 5 个）

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
print("Drive mounted at /content/drive/MyDrive")

In [ ]:
from pathlib import Path

DATASET_ROOT = Path("/content/drive/MyDrive/dataset")
MAX_SUBDIRS = 5
MAX_FILES_SAMPLE = 8
MAX_DEPTH = 2  # 每个子文件夹再往下扫一层


def _fmt_size(n: int) -> str:
    for u, div in (("GB", 1 << 30), ("MB", 1 << 20), ("KB", 1 << 10)):
        if n >= div:
            return f"{n / div:.1f}{u}"
    return f"{n}B"


def _list_dir(p: Path, prefix: str = "") -> None:
    if not p.is_dir():
        print(f"{prefix}[missing dir] {p}")
        return
    try:
        entries = sorted(p.iterdir(), key=lambda x: (not x.is_dir(), x.name.lower()))
    except PermissionError as e:
        print(f"{prefix}[permission denied] {p} ({e})")
        return
    dirs = [e for e in entries if e.is_dir()]
    files = [e for e in entries if e.is_file()]
    print(f"{prefix}dirs={len(dirs)} files={len(files)}")
    for f in files[:MAX_FILES_SAMPLE]:
        try:
            sz = _fmt_size(f.stat().st_size)
        except OSError:
            sz = "?"
        print(f"{prefix}  [file] {f.name}  ({sz})")
    if len(files) > MAX_FILES_SAMPLE:
        print(f"{prefix}  ... +{len(files) - MAX_FILES_SAMPLE} more files")
    for d in dirs[:MAX_SUBDIRS]:
        print(f"{prefix}  [dir] {d.name}/")
    if len(dirs) > MAX_SUBDIRS:
        print(f"{prefix}  ... +{len(dirs) - MAX_SUBDIRS} more subdirs (skipped)")


def _scan_subtree(p: Path, depth: int, prefix: str) -> None:
    if depth <= 0:
        return
    try:
        subdirs = sorted([x for x in p.iterdir() if x.is_dir()], key=lambda x: x.name.lower())
    except (PermissionError, OSError) as e:
        print(f"{prefix}[error] {e}")
        return
    for sd in subdirs[:MAX_SUBDIRS]:
        print(f"{prefix}└─ {sd.name}/")
        _list_dir(sd, prefix + "    ")
        _scan_subtree(sd, depth - 1, prefix + "    ")


print("=" * 72)
print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"exists={DATASET_ROOT.exists()}  is_dir={DATASET_ROOT.is_dir()}")
print("=" * 72)

if not DATASET_ROOT.is_dir():
    parent = DATASET_ROOT.parent
    if parent.is_dir():
        print(f"\n[hint] {DATASET_ROOT} 不存在。MyDrive 下可见项（前 20 个）：")
        for i, x in enumerate(sorted(parent.iterdir(), key=lambda p: p.name.lower())):
            if i >= 20:
                print("  ...")
                break
            tag = "dir" if x.is_dir() else "file"
            print(f"  [{tag}] {x.name}")
else:
    top = sorted([x for x in DATASET_ROOT.iterdir()], key=lambda p: p.name.lower())
    print(f"\ntop_level_datasets: {len(top)}\n")
    for i, ds in enumerate(top, 1):
        print("-" * 72)
        if ds.is_file():
            try:
                sz = _fmt_size(ds.stat().st_size)
            except OSError:
                sz = "?"
            print(f"[{i}/{len(top)}] [FILE at root] {ds.name}  ({sz})")
            continue
        print(f"[{i}/{len(top)}] {ds.name}/")
        _list_dir(ds, prefix="  ")
        _scan_subtree(ds, MAX_DEPTH, "  ")
        print()
    print("=" * 72)
    print("done")